# 04 — The estimator AdaBoostClassifier and its parameters

*Chapter 07 — AdaBoost · Notebook 4 of 5*

Three notebooks built AdaBoost by hand and watched how it behaves. This one is short and practical: we
meet scikit-learn's `AdaBoostClassifier` and turn its dials — knowing, from everything before, exactly
what each one does. There is no new magic here; every parameter maps to something you already understand.

**Prerequisites.**
- **Notebooks 1–3:** the by-hand reweighting loop and its parity with the library, the additive model,
  and `learning_rate` & overfitting behaviour.
- **Module 00 (NB 10):** cross-validation and `GridSearchCV` — tune on the training set, judge once on a
  sealed test.
- **Chapter 06:** feature importance (MDI) and its bias.

**What you'll be able to do.**
- Read `AdaBoostClassifier`'s constructor and set its parameters with intent.
- Set the **base learner** and explain why boosting needs it to stay **weak**.
- Tune `n_estimators` and `learning_rate` together, honestly, by cross-validation.
- Read `feature_importances_` with the right caveat, and state the current-API facts (SAMME only).

## Where we are

You have already done the hard part. The reweighting loop, the vote weight α, the additive model, the
learning rate, the overfitting behaviour — all built and measured by hand. So this notebook is a tour of
the estimator's knobs, not a derivation. The one genuinely new idea is practical: which base learner to
hand AdaBoost, and why the answer is "a weak one".

In [ ]:
import inspect

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import CMAP_COUNT, COLORS

viz.use_course_style()
SEED = 0

X, y = make_moons(n_samples=400, noise=0.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
print(f"train: {X_train.shape[0]} points    test: {X_test.shape[0]} points\n")

print("AdaBoostClassifier signature:")
print(" ", inspect.signature(AdaBoostClassifier.__init__))
fitted = AdaBoostClassifier(random_state=SEED).fit(X_train, y_train)
print("default base estimator:", fitted.estimator_)
print("has an 'algorithm' parameter:",
      "algorithm" in inspect.signature(AdaBoostClassifier.__init__).parameters)

**Read it.** Four parameters carry the method:

- **`estimator`** — the weak learner to boost. The default is a depth-1 decision tree (a **stump**), the
  canonical weak learner from notebook 1.
- **`n_estimators`** — how many rounds (weak learners) to add.
- **`learning_rate`** — the shrinkage ν from notebook 3, scaling every round's contribution.
- **`random_state`** — the seed.

One current-API note. Older scikit-learn had an `algorithm` parameter offering `SAMME` and a real-valued
variant `SAMME.R`; in version 1.9 that parameter is **gone** — only **SAMME** remains, exactly the
algorithm we derived in notebooks 1–2. There is no longer a variant to choose between.

In [ ]:
ada50 = AdaBoostClassifier(n_estimators=50, random_state=SEED).fit(X_train, y_train)
print(f"AdaBoost(50) test accuracy = {ada50.score(X_test, y_test):.4f}")
print(f"estimator_weights_[:3]     = {np.round(ada50.estimator_weights_[:3], 4)}")
print("(these are exactly the by-hand alpha from notebooks 1-2)")

**Read it.** The library estimator *is* the loop you wrote: the same learner weights
(`estimator_weights_` are our α), the same predictions, the same 0.9417 on the test set. From here we
stop looking inside and start turning the dials.

## The dial that matters most: the base learner

Boosting's whole bet (notebook 1) is that many **weak** learners, each correcting the last, add up to a
strong one. So the base learner should stay weak — a stump. What if we hand AdaBoost a *stronger* base, a
deeper tree? Intuition says a stronger base needs fewer rounds; let us see what it does to
generalization.

In [ ]:
print("base learner depth -> train / test accuracy (AdaBoost, 200 rounds)")
for depth in (1, 2, 3, 5):
    base = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    m = AdaBoostClassifier(estimator=base, n_estimators=200, random_state=SEED)
    m.fit(X_train, y_train)
    print(f"  max_depth={depth}:  train {m.score(X_train, y_train):.4f}   "
          f"test {m.score(X_test, y_test):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for depth, ax in zip((1, 3), axes, strict=False):
    base = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    m = AdaBoostClassifier(estimator=base, n_estimators=200, random_state=SEED)
    m.fit(X_train, y_train)
    viz.plot_decision_boundary(m, X_train, y_train, ax=ax)
    ax.set_title(f"base max_depth={depth}   (test {m.score(X_test, y_test):.3f})")
plt.show()

**Read the figure.** Two things. First, *every* base depth drives the **training** accuracy to
1.000 — with 200 rounds, AdaBoost memorizes the training set whatever base you give it. Second, and
decisively, the **test** accuracy *falls* as the base deepens: 0.9417 with a stump, down to 0.9167 with
a depth-5 base. The right panel makes it visible — the depth-3 base carves a more contorted boundary than
the stump's clean staircase, fitting wrinkles the stump left alone.

This is the **mirror image of a random forest** (chapter 06). A forest wants *deep*, low-bias trees and
averages away their variance. Boosting wants *weak*, high-bias learners and reduces bias by adding many
of them — a strong base learner overfits before the ensemble can help. Keep the default stump (or close
to it); it is the single most important setting in AdaBoost.

## Rounds and step size, together

`n_estimators` and `learning_rate` are not independent (notebook 3): a smaller step needs more rounds to
cover the same ground. So we tune them **together**, by cross-validation on the training set — never by
peeking at the test set.

In [ ]:
n_grid = [50, 100, 200, 400]
lr_grid = [0.1, 0.5, 1.0]
grid = GridSearchCV(
    AdaBoostClassifier(random_state=SEED),
    {"n_estimators": n_grid, "learning_rate": lr_grid},
    cv=5, scoring="accuracy", n_jobs=-1,
)
grid.fit(X_train, y_train)

# Pivot the CV results into a (learning_rate x n_estimators) table, keyed on the actual
# parameter values -- so the orientation can never drift from the data.
cv_table = pd.DataFrame(grid.cv_results_).pivot_table(
    index="param_learning_rate", columns="param_n_estimators", values="mean_test_score"
)
print("mean 5-fold CV accuracy (rows = learning_rate, cols = n_estimators):")
print(cv_table.round(3).to_string())
print(f"\nbest params: {grid.best_params_}   best CV accuracy: {grid.best_score_:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
im = ax.imshow(cv_table.values, cmap=CMAP_COUNT, aspect="auto", vmin=0.90, vmax=0.97,
               origin="lower")
ax.set_xticks(range(len(cv_table.columns)), [str(c) for c in cv_table.columns])
ax.set_yticks(range(len(cv_table.index)), [str(r) for r in cv_table.index])
ax.set_xlabel("n_estimators")
ax.set_ylabel("learning_rate")
for i in range(cv_table.shape[0]):
    for j in range(cv_table.shape[1]):
        ax.text(j, i, f"{cv_table.values[i, j]:.3f}", ha="center", va="center",
                color=COLORS["text"])
fig.colorbar(im, ax=ax, label="mean 5-fold CV accuracy")
plt.show()

**Read the figure.** The bottom-left corner — few rounds and small steps (`n_estimators` 50,
`learning_rate` 0.1) — **underfits** at ~0.91: the model has not taken enough steps to fit the data.
Accuracy rises with more rounds and/or a larger step, to a broad **0.95–0.96 plateau** covering the rest
of the map. The notebook-3 trade-off is visible directly: at `learning_rate` 0.1 the model needs all 400
rounds to reach 0.954, while `learning_rate` 1.0 is already at 0.957 by 100 rounds. The best cell is
`learning_rate` 0.5, `n_estimators` 400 (0.961) — but look how *flat* the plateau is. Like the random
forest of chapter 06, AdaBoost is **forgiving**: once it has enough rounds, the exact setting barely
matters.

In [ ]:
default = AdaBoostClassifier(random_state=SEED).fit(X_train, y_train)  # n=50, lr=1.0
tuned = grid.best_estimator_
default_cv = cv_table.loc[1.0, 50]
default_test = default.score(X_test, y_test)
tuned_test = tuned.score(X_test, y_test)
print(f"default (n=50, lr=1.0):  CV {default_cv:.4f}   test {default_test:.4f}")
print(f"tuned {grid.best_params_}:  CV {grid.best_score_:.4f}   test {tuned_test:.4f}")

In [ ]:
bp = grid.best_params_
labels = ["default\n(n=50, lr=1.0)", f"tuned\n(n={bp['n_estimators']}, lr={bp['learning_rate']})"]
xpos = np.arange(2)
width = 0.36
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(xpos - width / 2, [default_cv, grid.best_score_], width,
       color=COLORS["train"], label="CV accuracy (train)")
ax.bar(xpos + width / 2, [default_test, tuned_test], width,
       color=COLORS["test"], label="sealed test accuracy")
ax.set_xticks(xpos, labels)
ax.set_ylim(0.90, 0.97)
ax.set_ylabel("accuracy")
ax.legend(loc="upper left")
for i, (cv, te) in enumerate(zip([default_cv, grid.best_score_], [default_test, tuned_test],
                                 strict=False)):
    ax.text(i - width / 2, cv + 0.001, f"{cv:.3f}", ha="center", fontsize=9)
    ax.text(i + width / 2, te + 0.001, f"{te:.3f}", ha="center", fontsize=9)
plt.show()

**Read the figure.** Cross-validation preferred the tuned model — its CV accuracy rose from 0.954
to 0.961. But on the **sealed test set, both models score exactly 0.9417**: the CV gain did **not**
transfer. On a problem this size, a fraction-of-a-point cross-validation difference is well within the
noise, and it would be a mistake to advertise the tuned model as "better". Chapter 06 found the same for
the random forest. The discipline still holds — tune on the training folds, then look at the test set
**once**, which is exactly what we did. The honest summary: AdaBoost with sensible defaults is already
near its best here.

In [ ]:
imp_model = AdaBoostClassifier(n_estimators=200, random_state=SEED).fit(X_train, y_train)
imp = imp_model.feature_importances_
print(f"feature_importances_ (MDI):  x1 = {imp[0]:.3f},  x2 = {imp[1]:.3f}   (sum {imp.sum():.3f})")

**Read it.** Like a tree or a forest, `AdaBoostClassifier` exposes `feature_importances_`: the
mean impurity decrease (MDI) each feature contributes across the stumps, weighted by their α. Here both
features carry real signal — the boundary needs both — with x1 counting a little more.

The **chapter-06 caveat carries over in full**: MDI is biased toward continuous and high-cardinality
features, and importance is **not** causation. With only two features there is little room for mischief;
on real data with dozens of features it matters, so the honest reading — MDI cross-checked with
permutation importance — waits for notebook 5. (For the record, multiclass needs no extra work: SAMME's
`+ ln(K − 1)` term from notebook 2 lets the same estimator handle more than two classes.)

## Honest scoping

- **The base learner must stay weak.** A deeper base overfits faster (measured: test 0.9417 → 0.9167 as
  depth goes 1 → 5); the default stump is the right choice — the opposite of a random forest's deep trees.
- **No `algorithm` / SAMME.R choice remains** — scikit-learn 1.9 ships SAMME only.
- **Tuning gains can be within noise.** Here cross-validation rose but the sealed test did not move; do
  not over-claim a fraction-of-a-point CV improvement.
- **`feature_importances_` is MDI** — biased and non-causal; notebook 5 reads importances honestly.
- **One sealed test, touched once** — all tuning was done on the training folds.

## Your turn

1. **The base depth (easy).** From Figure A and the sweep, which base depth generalizes best here? State
   in one sentence why boosting prefers that depth, and how that differs from what a random forest
   (chapter 06) prefers.
2. **Extend the grid (medium).** Add `learning_rate = 0.05` and `n_estimators = 800` to the `GridSearchCV`
   grid and re-run it. Does the plateau extend into the new corner, and does the best CV score move
   meaningfully?
3. **A stronger base, re-tuned (harder).** Set `estimator = DecisionTreeClassifier(max_depth=3)` and tune
   `n_estimators` alone. Does the stronger base need **fewer** rounds to peak, and does it beat the stump
   on the sealed test? Explain from the bias/variance picture.

## What you built

- You read `AdaBoostClassifier`'s constructor and set each dial with intent — confirming the library
  estimator reproduces your by-hand loop.
- You found the method's most important setting: the **base learner must stay weak** (a deeper base
  overfits faster), the mirror image of a random forest.
- You tuned `n_estimators` and `learning_rate` **together** by cross-validation, read the forgiving
  plateau, and saw honestly that the CV gain did **not** beat the default on the sealed test.
- You read `feature_importances_` (MDI) with the chapter-06 caveat, and noted SAMME handles multiclass
  for free.

**Vocabulary you now own:** `estimator` (the base learner) and its strength · `n_estimators` ·
`learning_rate` · SAMME-only (`algorithm` removed) · `GridSearchCV` (tune on train, one sealed test) ·
`feature_importances_` / MDI.

## References

- Freund, Y., & Schapire, R. E. (1997). *A decision-theoretic generalization of on-line learning and an
  application to boosting.* Journal of Computer and System Sciences 55(1), 119–139.
  DOI: [10.1006/jcss.1997.1504](https://doi.org/10.1006/jcss.1997.1504)
- Zhu, J., Zou, H., Rosset, S., & Hastie, T. (2009). *Multi-class AdaBoost.* Statistics and Its Interface
  2(3), 349–360. DOI: [10.4310/SII.2009.v2.n3.a8](https://doi.org/10.4310/SII.2009.v2.n3.a8)
- Pedregosa, F., et al. (2011). *Scikit-learn: Machine Learning in Python.* Journal of Machine Learning
  Research 12, 2825–2830.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10. DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)

*Previous: 03 — Learning rate, number of rounds, and overfitting behaviour. Next: 05 — A demanding case:
spambase.*